# Notebook 3 — Random Forest Classification

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indicium15/ml-workshop/blob/main/workshop-2/notebooks/03_Random_Forest_Classification.ipynb)

**Best used for:** predicting a known category and identifying useful predictive features.

## What is a Random Forest?

A **Random Forest** is an ensemble of decision trees. Each tree is trained on a random subset of the training data (**bagging**) and considers only a random subset of features at each split. The final prediction is the **majority vote** across all trees.

Key ideas:
- **Decision trees** make predictions by splitting data on feature thresholds.
- **Bagging** reduces variance by averaging many noisy trees.
- **Feature importance** measures how much each feature reduces prediction error across all trees.

## Connection to the Clustering Notebooks

In Notebooks 1 and 2, we discovered *student archetypes* without using any labels. We used `performance_band` only as a sanity check. Now we ask:

> **"Can we predict the performance band we defined earlier?"**

This is a **supervised classification** task — we train on labelled examples and measure how accurately the model classifies new students.

---

## Running This Notebook

You can use this notebook in either:

- **Google Colab**: best if you do not have Python installed.
- **Local Jupyter**: best if you already have the workshop folder on your computer.

### In Google Colab

1. Open this notebook from GitHub using the **Open in Colab** button.
2. Run the first setup cell.
3. Wait until it says the libraries and workshop files are loaded.
4. In **Step 1**, either use the pre-loaded sample dataset or upload your own CSV file.
5. Continue running each cell from top to bottom.

If a widget does not appear immediately in Colab, wait for the setup cell to finish, then rerun the current cell.

In [ ]:
# SETUP
import sys, os, subprocess, importlib.util
from pathlib import Path

# Works locally from workshop-2/notebooks, from workshop-2, or in Google Colab
# after opening the notebook from GitHub.
REPO_URL = 'https://github.com/indicium15/ml-workshop.git'

def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / 'workshop-2',
        Path.cwd().parent / 'workshop-2',
        Path('/content/ml-workshop/workshop-2'),
        Path('/content/workshop-2'),
    ]
    for candidate in candidates:
        if (candidate / 'utils' / 'data_loader.py').exists():
            return candidate.resolve()
    return None

IN_COLAB = 'google.colab' in sys.modules
WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None and IN_COLAB:
    print('Workshop files not found in Colab. Cloning the workshop repository...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/ml-workshop'], check=True)
    WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None:
    raise FileNotFoundError(
        'Could not find workshop-2/utils. Locally, start Jupyter from the workshop-2 folder. '
        'In Colab, upload the workshop-2 folder or open the notebook from the GitHub repository.'
    )

if str(WORKSHOP_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSHOP_ROOT))

required_modules = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'scipy': 'scipy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'ipywidgets': 'ipywidgets',
}
missing = [pip_name for module_name, pip_name in required_modules.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORKSHOP_ROOT / 'requirements.txt')], check=True)

import ipywidgets as widgets

print(f'Using workshop files from: {WORKSHOP_ROOT}')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.tree import plot_tree

from utils.data_loader import DataLoaderWidget
from utils.preprocessor import PreprocessorWidget
from utils.plotting import (
    plot_confusion_matrix,
    plot_feature_importance,
)

print('Libraries loaded ✓')


### Colab Notes

- The first setup cell may take a minute because it checks for required packages.
- Uploaded files in Colab are temporary. If the runtime disconnects, upload the file again.
- To keep your own completed version, use **File → Save a copy in Drive**.
- If widgets do not display, run the setup cell again, then rerun the current cell.

## Step 1 — Load Data

The sample dataset is already available when you run the setup cell.

To use your own data in Colab:

1. Click the upload control.
2. Choose a `.csv` file from your computer.
3. Wait for the preview table to appear.
4. Select the columns you want to use.
5. Click **Confirm Selection**.

For local Jupyter, you can either upload a CSV or type a path to a CSV file on your machine.

### What the Load Data controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **CSV path** | Points the notebook to a CSV file on disk. The default path loads the workshop sample dataset. | Change this when running locally with your own CSV file. |
| **Load CSV** | Reads the file from **CSV path** and displays a short preview plus column counts. | Click this after editing the path. |
| **Or upload** | Lets you upload a CSV directly in Colab/Jupyter instead of typing a path. | Use this when your CSV is on your computer rather than already in the notebook folder. |
| **Features** | Selects the input columns the later analysis/model will use. Hold Ctrl/Cmd to select multiple columns. | Exclude IDs, labels/targets, free-text notes, or columns you do not want the method to learn from. |
| **Select all** | Selects every column except common ID/target columns. | Useful as a starting point, then remove anything inappropriate. |
| **Numeric only** | Selects only integer/float columns. | Useful for a simpler first run or for methods where you want to avoid categorical encoding. |
| **Label col** | Chooses the target column to predict. | Use the column that contains the known category/outcome, such as `performance_band`. |
| **Confirm Selection** | Saves the selected features and label column for later cells. | Click this before preprocessing or running any analysis. |

> Keep a copy of your original dataset unchanged. These notebooks do not edit the source CSV.

> **Tip:** Select `performance_band` as the label column for the sample dataset. With your own data, choose a categorical column that represents the outcome you want to predict.


In [ ]:
# LOAD DATA
loader = DataLoaderWidget(show_label_selector=True)
loader.display()

## Step 2 — Preprocessing

Choose how to handle categorical features and how much data to reserve for testing.

### What the Preprocessing controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **Scaling** | Rescales numeric features. Random Forests do not require scaling, but scaling is available so the workflow stays consistent with the clustering notebooks. | Leaving `standard` is fine for the workshop. `none` is also acceptable for tree-based models if you want raw feature values preserved. |
| **Categorical enc** | Converts text/category columns into model-ready numeric columns. `label` assigns each category an integer; `onehot` creates one column per category; `drop` removes categorical columns. | Use `onehot` when category order should not be implied. Use `drop` if a categorical field is noisy or not available in real prediction use. |
| **Test split** | Reserves a percentage of rows for evaluation instead of training. | Use 20% as a standard starting point; increase it for larger datasets or reduce it when the dataset is very small. |
| **Random state** | Controls the train/test split randomness so results can be reproduced. | Keep the default when comparing settings. Change it to see whether results are stable across different splits. |
| **Apply Preprocessing** | Creates the training and test sets used by the Random Forest model. | Click after changing selected columns, label column, or preprocessing settings. |



In [ ]:
# HYPERPARAMETERS
preprocessor = PreprocessorWidget(
    source_loader=loader,
    clustering_mode=False,
)
preprocessor.display()


## Step 3 — Feature Exclusion / Leakage Check

Some datasets contain columns that make prediction unrealistically easy because they directly encode the answer, were created from the answer, or would not be available at prediction time. Use this configurable selector to remove those columns before training.

### What the Leakage Check controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **Flag words** | Comma-separated words used to search feature names for possible leakage columns. | Add words that often identify target-like fields in your dataset, such as `score`, `grade`, `result`, or `band`. |
| **Remove** | Selects columns to exclude from model training. | Select columns that directly reveal the answer, are IDs, or would not be available at prediction time. |
| **Refresh Feature List** | Rebuilds the removable-feature list from the preprocessed training columns. | Click after changing feature selection or preprocessing. |
| **Suggest Columns to Review** | Finds columns whose names match the flag words and selects them for review. | Use it as a first pass, then manually keep only the columns that truly should be removed. |
| **Clear Selection** | Deselects all columns in **Remove**. | Use it when you want to start the leakage review again. |



In [ ]:
# HYPERPARAMETERS
_leakage_out = widgets.Output()
_pattern_text = widgets.Text(
    value='target,label,grade,band,score,final,average,avg,total,result,outcome',
    description='Flag words:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='620px'),
)
_drop_cols = widgets.SelectMultiple(
    options=[],
    value=(),
    description='Remove:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='620px', height='180px'),
)
_refresh_leakage_btn = widgets.Button(
    description='Refresh Feature List',
    button_style='',
    icon='refresh',
)
_suggest_leakage_btn = widgets.Button(
    description='Suggest Columns to Review',
    button_style='warning',
    icon='search',
)
_clear_drop_btn = widgets.Button(
    description='Clear Selection',
    button_style='',
    icon='times',
)

def _feature_columns_for_training():
    if preprocessor.X_train is not None:
        return list(preprocessor.X_train.columns)
    if loader.X_df is not None:
        return list(loader.X_df.columns)
    return []

def _selected_drop_columns():
    cols = set(_feature_columns_for_training())
    return [c for c in _drop_cols.value if c in cols]

def _refresh_leakage_columns(_btn=None):
    cols = _feature_columns_for_training()
    _drop_cols.options = cols
    _drop_cols.value = tuple(c for c in _drop_cols.value if c in cols)
    _leakage_out.clear_output()
    with _leakage_out:
        if not cols:
            display(widgets.HTML('<span style="color:red">Load and preprocess data first, then refresh this list.</span>'))
        else:
            display(widgets.HTML(
                f'<span style="color:green">✓ Found {len(cols)} model feature columns. '
                'Select any columns that should be excluded before training.</span>'
            ))

def _suggest_leakage_columns(_btn):
    cols = _feature_columns_for_training()
    patterns = [p.strip().lower() for p in _pattern_text.value.split(',') if p.strip()]
    target_name = getattr(loader, '_label_dropdown', None)
    target_value = target_name.value.lower() if target_name is not None and target_name.value else ''
    suggestions = []
    for col in cols:
        low = col.lower()
        if target_value and target_value in low:
            suggestions.append(col)
        elif any(pattern in low for pattern in patterns):
            suggestions.append(col)
    _drop_cols.options = cols
    _drop_cols.value = tuple(suggestions)
    _leakage_out.clear_output()
    with _leakage_out:
        if suggestions:
            display(widgets.HTML('<b>Suggested columns to review:</b>'))
            display(pd.DataFrame({'column': suggestions}))
            display(widgets.HTML('<span style="color:#555">Keep only the columns you truly want removed selected above.</span>'))
        else:
            display(widgets.HTML('<span style="color:green">No column names matched the flag words.</span>'))

def _clear_drop_columns(_btn):
    _drop_cols.value = ()

_refresh_leakage_btn.on_click(_refresh_leakage_columns)
_suggest_leakage_btn.on_click(_suggest_leakage_columns)
_clear_drop_btn.on_click(_clear_drop_columns)

display(widgets.VBox([
    widgets.HTML('<h3>🔍 Step 3 — Feature Exclusion / Leakage Check</h3>'),
    _pattern_text,
    widgets.HBox([_refresh_leakage_btn, _suggest_leakage_btn, _clear_drop_btn]),
    _drop_cols,
    _leakage_out,
]))


## Step 4 — Random Forest Hyperparameters

| Parameter | Effect | When to adjust it |
|---|---|---|
| **N estimators** | Number of trees in the forest. More trees usually reduce variance but increase training time. | 100-200 is a good starting range; increase for stability if training time is acceptable. |
| **Unlimited depth** | Lets each tree grow until stopping rules are met. | Turn off when the model overfits or you want simpler trees. |
| **Max depth** | Maximum depth of each tree when unlimited depth is off. | Lower values reduce overfitting; higher values allow more complex patterns. |
| **Min samples split** | Minimum number of samples required to split an internal node. | Increase it to regularise the model and avoid splits based on tiny groups. |
| **Min samples leaf** | Minimum number of samples allowed in a final leaf. | Increase it for smoother, less brittle predictions. |
| **Max features** | Number of candidate features considered at each split. | `sqrt` is a strong default for classification; try `log2` or `None` when comparing model behavior. |
| **Class weight** | Adjusts weighting for imbalanced target classes. | Use `balanced` when minority classes are being ignored. |
| **Random state** | Controls reproducible bootstrapping and feature sampling. | Keep fixed when comparing settings; change it to test stability. |
| **Train Model** | Fits the Random Forest and evaluates it on the test set. | Click after changing preprocessing, leakage exclusions, or hyperparameters. |

> **Overfitting warning:** If train accuracy is much higher than test accuracy (gap > 15%), the model is memorising the training data. Try reducing `max_depth` or increasing `min_samples_leaf`.



In [ ]:
# HYPERPARAMETERS
_rf_out = widgets.Output()
_rf_model = None

_n_estimators = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description='N estimators:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px'),
)
_unlimited_depth = widgets.Checkbox(
    value=False,
    description='Unlimited depth',
    layout=widgets.Layout(width='200px'),
)
_max_depth = widgets.IntSlider(
    value=10, min=2, max=20, step=1,
    description='Max depth:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px'),
)
_min_samples_split = widgets.IntSlider(
    value=2, min=2, max=20, step=1,
    description='Min samples split:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px'),
)
_min_samples_leaf = widgets.IntSlider(
    value=1, min=1, max=10, step=1,
    description='Min samples leaf:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px'),
)
_max_features_dd = widgets.Dropdown(
    options=['sqrt', 'log2', 'None'],
    value='sqrt',
    description='Max features:',
    style={'description_width': '150px'},
)
_class_weight_dd = widgets.Dropdown(
    options=['None', 'balanced'],
    value='None',
    description='Class weight:',
    style={'description_width': '150px'},
)
_rf_seed = widgets.IntText(
    value=42,
    description='Random state:',
    layout=widgets.Layout(width='200px'),
    style={'description_width': '110px'},
)
_train_rf_btn = widgets.Button(
    description='Train Model ▶',
    button_style='success',
    icon='play',
)

def _toggle_depth(change):
    _max_depth.disabled = _unlimited_depth.value

_unlimited_depth.observe(_toggle_depth, names='value')

def _train_rf(_btn):
    global _rf_model
    _rf_out.clear_output()
    with _rf_out:
        try:
            X_train = preprocessor.X_train
            X_test = preprocessor.X_test
            y_train = preprocessor.y_train
            y_test = preprocessor.y_test

            if X_train is None:
                display(widgets.HTML('<span style="color:red">Run preprocessing first.</span>'))
                return

            # Apply user-selected feature exclusions
            drop_cols = _selected_drop_columns() if '_selected_drop_columns' in globals() else []
            if drop_cols:
                X_train = X_train.drop(columns=drop_cols, errors='ignore')
                X_test = X_test.drop(columns=drop_cols, errors='ignore')
                display(widgets.HTML(f'<span style="color:blue">ℹ Removed from training: {drop_cols}</span>'))

            max_depth = None if _unlimited_depth.value else _max_depth.value
            max_features = None if _max_features_dd.value == 'None' else _max_features_dd.value
            class_weight = None if _class_weight_dd.value == 'None' else _class_weight_dd.value

            _rf_model = RandomForestClassifier(
                n_estimators=_n_estimators.value,
                max_depth=max_depth,
                min_samples_split=_min_samples_split.value,
                min_samples_leaf=_min_samples_leaf.value,
                max_features=max_features,
                class_weight=class_weight,
                random_state=_rf_seed.value,
                n_jobs=-1,
            )
            display(widgets.HTML('Training... (this may take a moment for large N estimators)'))
            _rf_model.fit(X_train, y_train)

            y_pred_train = _rf_model.predict(X_train)
            y_pred_test = _rf_model.predict(X_test)
            train_acc = accuracy_score(y_train, y_pred_train)
            test_acc = accuracy_score(y_test, y_pred_test)

            overfit_msg = ''
            if train_acc - test_acc > 0.15:
                overfit_msg = (' <span style="color:orange">⚠ Large train-test gap — model may be overfitting. '
                               'Try reducing max_depth or increasing min_samples_leaf.</span>')

            display(widgets.HTML(
                f'<b>Train accuracy: {train_acc:.4f} &nbsp;|&nbsp; '
                f'Test accuracy: {test_acc:.4f}</b>{overfit_msg}'
            ))

            # Classification report
            report_dict = classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)
            report_df = pd.DataFrame(report_dict).T
            display(widgets.HTML('<b>Classification Report:</b>'))
            display(report_df.style.format('{:.3f}').background_gradient(cmap='YlGn', axis=None))

            # Confusion matrix
            labels_sorted = sorted(set(list(y_test) + list(y_pred_test)))
            fig = plot_confusion_matrix(y_test, y_pred_test, labels=labels_sorted)
            plt.show()

        except Exception as exc:
            import traceback
            display(widgets.HTML(f'<span style="color:red">Error: {exc}<br><pre>{traceback.format_exc()}</pre></span>'))

_train_rf_btn.on_click(_train_rf)
display(widgets.VBox([
    widgets.HTML('<h3>🌲 Step 4 — Random Forest Hyperparameters</h3>'),
    _n_estimators,
    widgets.HBox([_unlimited_depth, _max_depth]),
    _min_samples_split,
    _min_samples_leaf,
    _max_features_dd,
    _class_weight_dd,
    _rf_seed,
    _train_rf_btn,
]))
display(_rf_out)

## Step 5 — Feature Importance

Which features does the Random Forest consider most informative?

- **Built-in importance** (Mean Decrease in Impurity): fast, but can favour high-cardinality features.
- **Permutation importance**: more reliable because it measures how much accuracy drops when each feature is randomly shuffled.

### What the Feature Importance controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **Show top N** | Chooses how many features appear in the importance chart. | Increase it for a fuller ranking; decrease it for a cleaner chart. |
| **Use permutation importance** | Computes importance by shuffling one feature at a time and measuring performance drop. Slower but often more trustworthy. | Enable it when you want a more reliable explanation and can wait longer. |
| **Plot Feature Importance** | Creates the feature-importance chart and table. | Click after training the model or changing importance settings. |



In [ ]:
# OUTPUT
_fi_out = widgets.Output()

_top_n_slider = widgets.IntSlider(
    value=10, min=5, max=20, step=1,
    description='Show top N:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='350px'),
)
_perm_check = widgets.Checkbox(
    value=False,
    description='Use permutation importance (slower but more reliable)',
    layout=widgets.Layout(width='400px'),
)
_fi_btn = widgets.Button(
    description='Plot Feature Importance',
    button_style='info',
    icon='bar-chart',
)

def _plot_fi(_btn):
    _fi_out.clear_output()
    with _fi_out:
        try:
            if _rf_model is None:
                display(widgets.HTML('<span style="color:red">Train the model first (Step 4).</span>'))
                return

            X_train = preprocessor.X_train
            X_test = preprocessor.X_test
            y_test = preprocessor.y_test

            drop_cols = _selected_drop_columns() if '_selected_drop_columns' in globals() else []
            if drop_cols:
                X_train = X_train.drop(columns=drop_cols, errors='ignore')
                X_test = X_test.drop(columns=drop_cols, errors='ignore')

            feat_names = list(X_train.columns)
            top_n = _top_n_slider.value

            if _perm_check.value:
                display(widgets.HTML('Computing permutation importance (may take ~30s)...'))
                perm = permutation_importance(
                    _rf_model, X_test, y_test,
                    n_repeats=10, random_state=42, n_jobs=-1
                )
                importances = perm.importances_mean
                title = f'Permutation Feature Importance — top {top_n}'
            else:
                importances = _rf_model.feature_importances_
                title = f'Feature Importance (MDI) — top {top_n}'

            fig = plot_feature_importance(
                importances,
                feat_names,
                top_n=top_n,
                title=title,
            )
            plt.show()

            # Attendance comparison
            if 'tutorial_attendance_rate' in feat_names and 'lecture_attendance_rate' in feat_names:
                t_idx = feat_names.index('tutorial_attendance_rate')
                l_idx = feat_names.index('lecture_attendance_rate')
                t_imp = importances[t_idx]
                l_imp = importances[l_idx]
                ratio = t_imp / l_imp if l_imp > 0 else float('inf')
                display(widgets.HTML(
                    f'<b>Attendance comparison:</b> '
                    f'tutorial_attendance_rate = {t_imp:.4f}, '
                    f'lecture_attendance_rate = {l_imp:.4f} '
                    f'(tutorial is <b>{ratio:.1f}×</b> more important)'
                ))

        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_fi_btn.on_click(_plot_fi)
display(widgets.VBox([
    widgets.HTML('<h3>📊 Step 5 — Feature Importance</h3>'),
    _top_n_slider,
    _perm_check,
    _fi_btn,
    _fi_out,
]))

## Step 6 — Optional: Visualise a Decision Tree

This section shows one shallow tree from the forest so you can inspect the kind of rules the model learned. It is intentionally limited to depth 3 so the plot remains readable.

### What the Tree Visualisation controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **Visualise one decision tree** | Enables the optional tree plot. | Turn it on when you want an interpretable example of one tree's decision rules. |
| **Show Tree** | Displays a single tree from the trained Random Forest, capped at depth 3. | Click after training the model and enabling the checkbox. |



In [ ]:
# OUTPUT
_tree_out = widgets.Output()
_viz_tree_check = widgets.Checkbox(
    value=False,
    description='Visualise one decision tree (max depth 3)',
    layout=widgets.Layout(width='360px'),
)
_tree_btn = widgets.Button(
    description='Show Tree',
    button_style='',
    icon='sitemap',
)

def _show_tree(_btn):
    _tree_out.clear_output()
    with _tree_out:
        try:
            if _rf_model is None:
                display(widgets.HTML('<span style="color:red">Train the model first.</span>'))
                return
            if not _viz_tree_check.value:
                display(widgets.HTML('<span style="color:orange">Enable the checkbox above first.</span>'))
                return

            X_train = preprocessor.X_train
            drop_cols = _selected_drop_columns() if '_selected_drop_columns' in globals() else []
            if drop_cols:
                X_train = X_train.drop(columns=drop_cols, errors='ignore')

            tree = _rf_model.estimators_[0]
            fig, ax = plt.subplots(figsize=(20, 8))
            plot_tree(
                tree,
                max_depth=3,
                feature_names=list(X_train.columns),
                class_names=_rf_model.classes_,
                filled=True,
                rounded=True,
                fontsize=9,
                ax=ax,
            )
            ax.set_title('Decision Tree #0 (shown at max depth 3 for readability)')
            plt.tight_layout()
            plt.show()
        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_tree_btn.on_click(_show_tree)
display(widgets.VBox([
    widgets.HTML('<h3>🌲 Step 6 — Tree Visualisation (Optional)</h3>'),
    _viz_tree_check,
    _tree_btn,
    _tree_out,
]))

## Reflection

Use your results above to answer the following questions:

---

**1. Which features were most predictive?**

*(Fill in from feature importance chart)*

Does this match your intuition from the HCA and K-Means notebooks?

---

**2. Was tutorial attendance more important than lecture attendance?**

The sample dataset was generated with `tutorial_attendance_rate` having a stronger correlation with performance. Does your model agree? Is this consistent with the education research literature?

---

**3. What does the model get wrong most often?**

Look at the confusion matrix. Are errors more common between:
- At-Risk vs Pass (similar performance level)?
- Merit vs Distinction?

What does this tell you about how the performance bands are defined?

---

**4. Overfitting check**

What was the gap between train and test accuracy? How did changing `max_depth` affect this?

---

**5. Model limitations**

- The model was trained on **synthetic data** — real student data would show different patterns.
- Feature importance does not imply causation — a feature being predictive does not mean increasing it will improve performance.
- Consider: what ethical questions arise if this model were deployed to identify at-risk students?

---

> **Workshop complete.** You have now experienced the full ML pipeline: unsupervised discovery (HCA → K-Means) → supervised prediction (Random Forest).

---

## After the Workshop: Reusing This Random Forest Notebook

Use this notebook when you have a label column and want to predict it from other columns.

Random Forest is useful for:

- classification tasks
- estimating which features are most predictive
- comparing train and test performance
- checking whether a model generalises beyond the training data

### What to Save

For your own analysis notes, save:

- the label column
- the feature columns used
- any excluded leakage columns
- train accuracy
- test accuracy
- confusion matrix
- classification report
- feature importance chart
- notes about likely overfitting

### Reuse Warning

A high accuracy score does not always mean the model is useful. Check for:

- leakage columns
- class imbalance
- very small datasets
- train accuracy much higher than test accuracy
- features that would not be available at prediction time